# v4 · M5 weekly (same records, T = 176)

## v4: why does BUS miss subtle time-series patterns, and can better mini-clusters fix it?
Questions from the "Pimp my BUS" project, answered on time-series data with **planted patterns in a null
background** (real target, permuted descriptors, effect = snr × null spread σ):

- **Q4 seeds.** Mini-clusters from five seed types:

  | seed | what a mini-cluster is |
  |---|---|
  | `closest-$` | today's BUS: the 200 closest pairs in excess-dollar space (convex, scale-dominated) |
  | `closest-rel` | closest pairs on the relative deviation (scale-free) |
  | `closest-blocks` | closest pairs on post-base relative deviation averaged into 36 time blocks (scale-free + denoised) |
  | `snn-pairs` | mutual nearest neighbours that share the most neighbours (density-based) |
  | `snn-components` | connected components of the shared-nearest-neighbour graph: **non-convex** chains, described by conditions shared by ≥ 60% of members |

  Per seed: purity, planted coverage, candidate recall and seed time.
- **Q1 pattern taxonomy:** level, trend, stockout (money, M2); seasonal wave, lag (shape, M3); volatility (variance, new M4).
- **Q3 local search:** from BUS's 10 best candidates, add / drop / swap one condition while quality improves.
- **Q2 meta-features:** dataset- and task-level features saved with every run, for the meta-analysis in `compare_v4.ipynb`.

In [1]:
import os
import time
import heapq
import itertools
from collections import Counter

import numpy as np
import pandas as pd
from scipy.spatial.distance import cdist
from joblib import Parallel, delayed
import matplotlib.pyplot as plt

pd.set_option("display.width", 250)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_colwidth", 110)
N_JOBS = os.cpu_count()
print("CPU cores:", N_JOBS)

CPU cores: 8


In [2]:
DATASET = "m5_weekly"
QUICK = False          # True = tiny grids, only to test the notebook end to end
UNIT = "$"
PERIOD = "week"
PERIODS_PER_YEAR = 52

# ---- Search parameters (Bart §6.3.1) ----
D_MAX = 3                # max conditions in a description
Q = 20                   # result set size
W = 50                   # default beam width
U = 200                  # mini-clusters (pairs) for every BUS variant and both ensembles
POOL_SIZE = 30           # pool ensemble: conditions the BS step may use
W_MC = 5                 # per-mini-cluster ensemble: beam width of each small beam search

# ---- Narrow findings ----
MIN_COV = 0.02
MAX_COV = 0.15
MIN_STORES = 2
MIN_PEERS = 5

# ---- Reference point ----
START_DATE, END_DATE = "2013-01-01", "2016-05-22"   # base window = all of 2013
MIN_UNITS_PER_DAY = 1.0     # keep item x store series selling at least this many units per day in 2013
M5_DIR = "/kaggle/input/m5-forecasting-accuracy" if "KAGGLE_KERNEL_RUN_TYPE" in os.environ else "M5"
RESOLUTION = "W"

# ---- Experiment 0: real data ----
MEASURES = ["M1", "M2", "M3"]
METHODS = ["BS", "BUS", "BUS-α", "BUS→BS pool", "BUS→BS per-MC"]
NU = 20 if QUICK else 500                     # permutation runs for the DFD (Bart used 500)

# ---- Experiment 1: planted subgroups ----
PATTERNS = {"level": "M2", "trend": "M2", "timing": "M3"}    # pattern -> the measure that should catch it
SNR = [2, 8] if QUICK else [1, 2, 4, 8]                      # planted effect in units of the null spread sigma (see E1)
LITS = [2] if QUICK else [2, 3]                               # conditions in the planted description
OUTLIERS = [0.0] if QUICK else [0.0, 0.2]                     # share of planted records left unchanged
REPS = 1 if QUICK else 2                                      # different planted descriptions per setting
PLANT_COVERAGE = (0.02, 0.08)
E1_METHODS = ["BS", "BS w=10", "BUS", "BUS-α", "BUS→BS pool", "BUS→BS per-MC"]

# ---- Experiment 2: interaction-only subgroups ----
E2_SNR = [8] if QUICK else [4, 8]
E2_REPS = 1 if QUICK else 3
E2_METHODS = ["BS w=5", "BS w=10", "BS w=20", "BS", "BUS", "BUS-α", "BUS→BS pool", "BUS→BS per-MC"]

HIT_JACCARD = 0.8        # a planted subgroup counts as found if a subgroup with Jaccard >= 0.8 was evaluated
SEED = 0
ON_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ
RESULTS_DIR = "/kaggle/working/results_v4" if ON_KAGGLE else "results_v4"
os.makedirs(RESULTS_DIR, exist_ok=True)

In [3]:
# ================= v4 experiment settings =================
# pattern -> the quality measure that matches it
PATTERNS_V4 = {"level": "M2", "trend": "M2", "stockout": "M2",      # money patterns (Konijn-style excess)
               "season": "M3", "lag": "M3",                           # shape patterns (Bart phi_cluster)
               "volatility": "M4"}                                    # variance pattern (new M4)
SNR_V4 = [2, 8] if QUICK else [2, 4, 8]           # planted strength in units of the null spread sigma
LAG_STEPS = {2: 1, 4: 2, 8: 4}                    # lag pattern: shift by this many LAG_UNITs instead of snr x sigma
LAG_UNIT = {"day": 7, "week": 1, "month": 1}[PERIOD]
LITS_V4 = [2] if QUICK else [2, 3]
REPS_V4 = 1 if QUICK else 3

# seeds (mini-clusters)
U_SEEDS = 200          # pairs per pair-based seed type
KNN = 10               # neighbours for the kNN / shared-nearest-neighbour graph
SNN_MIN = 5            # edge in the SNN graph if two mutual neighbours share >= SNN_MIN neighbours
Z_COMP_MIN = 3         # smallest non-convex component used as a mini-cluster
MAJORITY = 0.6         # a component describes itself by conditions shared by >= 60% of its members
N_BLOCKS = 36          # time blocks of the denoised representation
LOCAL_TOPK = 10        # Q3: local search starts from BUS's 10 best candidates

SEED_TYPES = ["closest-$", "closest-rel", "closest-blocks", "snn-pairs", "snn-components"]
METHODS_V4 = ["BS w=10", "BS",
              "BUS closest-$", "BUS→BS pool α-$",                       # the old BUS and old pool ensemble
              "BUS closest-rel", "BUS closest-blocks", "BUS snn-pairs", "BUS snn-components",
              "BUS→BS pool blocks", "BUS→BS pool snn",
              "BUS snn-pairs +local", "BUS closest-blocks +local"]
RESULTS_DIR = "/kaggle/working/results_v4" if ON_KAGGLE else "results_v4"
os.makedirs(RESULTS_DIR, exist_ok=True)

In [4]:
calendar = pd.read_csv(f"{M5_DIR}/calendar.csv", parse_dates=["date"]).sort_values("date").reset_index(drop=True)
sales_wide = pd.read_csv(f"{M5_DIR}/sales_train_evaluation.csv")
prices = pd.read_csv(f"{M5_DIR}/sell_prices.csv")

day_cols = [c for c in sales_wide.columns if c.startswith("d_")]
all_dates = calendar["date"].iloc[[int(c[2:]) - 1 for c in day_cols]].reset_index(drop=True)
in_period = (all_dates >= START_DATE) & (all_dates <= END_DATE)
dates = pd.DatetimeIndex(all_dates[in_period])
units = sales_wide[day_cols].to_numpy(dtype=np.float32)[:, in_period.to_numpy()]
BASE_LEN = int((dates < "2014-01-01").sum())

keep = units[:, :BASE_LEN].mean(axis=1) >= MIN_UNITS_PER_DAY
sales_wide, units = sales_wide[keep].reset_index(drop=True), units[keep]

# Dollars = units x the weekly shelf price of that item in that store (filled across weeks without a price).
week_of_day = calendar.set_index("date").loc[dates, "wm_yr_wk"].to_numpy()
weeks = np.unique(week_of_day)
price_wide = (prices.pivot_table(index=["store_id", "item_id"], columns="wm_yr_wk", values="sell_price")
                    .reindex(columns=weeks))
key = pd.MultiIndex.from_frame(sales_wide[["store_id", "item_id"]])
price_wide = price_wide.reindex(key).ffill(axis=1).bfill(axis=1)
price_daily = price_wide.to_numpy()[:, np.searchsorted(weeks, week_of_day)]
S = units.astype(float) * price_daily
if RESOLUTION == "W":                                    # same days, summed into 7-day blocks
    n_weeks = S.shape[1] // 7
    S = S[:, : n_weeks * 7].reshape(len(S), n_weeks, 7).sum(axis=2)
    dates = dates[: n_weeks * 7: 7]
    BASE_LEN = int((dates < "2014-01-01").sum())
MIN_BASE_LEVEL = 0.0       # the size filter was applied on units above

records = sales_wide[["store_id", "item_id", "dept_id", "cat_id", "state_id"]].rename(
    columns={"store_id": "store", "item_id": "item"})

# Item-level descriptors (same in every store): price level, price volatility, launch week.
item_prices = prices.groupby("item_id")["sell_price"]
first_week = prices.groupby("item_id")["wm_yr_wk"].min()
records["item_price"] = records["item"].map(item_prices.mean())
records["item_price_volatility"] = records["item"].map(item_prices.std() / item_prices.mean())
records["item_launch"] = records["item"].map(first_week.rank())

# Record-level descriptors (this store vs. the same item elsewhere): price level, price trend, discount frequency.
p = price_wide.to_numpy()
t_w = np.arange(p.shape[1], dtype=float)
t_w -= t_w.mean()
records["price_level"] = p.mean(axis=1)
records["price_trend"] = (p - p.mean(axis=1, keepdims=True)) @ t_w / (t_w ** 2).sum() * 52 / p.mean(axis=1)
records["discount_freq"] = (p < 0.95 * p.max(axis=1, keepdims=True)).mean(axis=1)

categorical_desc = [("state_id", "store"), ("cat_id", "item"), ("dept_id", "item")]
numeric_desc = [("item_price", "dept_id", "item"), ("item_price_volatility", None, "item"), ("item_launch", None, "item"),
                ("price_level", "item", "record"), ("price_trend", "item", "record"), ("discount_freq", "item", "record")]
del units, price_daily, sales_wide
print(S.shape, records.shape)

(8161, 176) (8161, 11)


In [5]:
level = S[:, :BASE_LEN].mean(axis=1)
item = records["item"].astype(str).to_numpy()

ok = level >= MIN_BASE_LEVEL
ok &= pd.Series(ok).groupby(item).transform("sum").to_numpy() >= MIN_PEERS
records = records[ok].reset_index(drop=True)
S, level, item = S[ok], level[ok], item[ok]
N, T = S.shape

group_sales = pd.DataFrame(S).groupby(item).transform("sum").to_numpy()
group_level = pd.Series(level).groupby(item).transform("sum").to_numpy()
peer_index = (group_sales - S) / (group_level - level)[:, None]
expected = level[:, None] * peer_index
excess = S - expected
rel_dev = np.clip(excess / level[:, None], -3, 3)
store_codes = pd.factorize(records["store"])[0]
del group_sales, peer_index

print(f"{DATASET}: N = {N} records, T = {T} {PERIOD}s, {records['store'].nunique()} stores, "
      f"{records['item'].nunique()} items, base window = {BASE_LEN} {PERIOD}s")
print(f"Base-window excess per record (0 by construction): max |.| = {np.abs(excess[:, :BASE_LEN].sum(axis=1)).max():.6f}")

m5_weekly: N = 6581 records, T = 176 weeks, 10 stores, 820 items, base window = 53 weeks
Base-window excess per record (0 by construction): max |.| = 0.000000


In [6]:
LABELS = ["very_low", "low", "medium", "high", "very_high"]


def z_bins(z):
    return pd.cut(z, bins=[-np.inf, -1.5, -0.5, 0.5, 1.5, np.inf], labels=LABELS).astype(str)


descriptor_cols, desc_level = [], {}
for column, level_ in categorical_desc:
    records[column] = records[column].astype(str)
    descriptor_cols.append(column)
    desc_level[column] = level_
for column, zgroup, level_ in numeric_desc:
    values = records[column].astype(float)
    if zgroup is None:
        z = (values - values.mean()) / values.std()
        name = f"{column}_cat"
    else:
        grouped = values.groupby(records[zgroup])
        z = (values - grouped.transform("mean")) / grouped.transform("std").replace(0, np.nan)
        name = f"{column}_vs_{zgroup}_cat"
    records[name] = z_bins(z.fillna(0))
    descriptor_cols.append(name)
    desc_level[name] = level_

A = records[descriptor_cols].astype(str).to_numpy(dtype=object)


def describe(desc):
    return " AND ".join(f"{descriptor_cols[j]} = {v}" for j, v in desc)


print(f"{len(descriptor_cols)} descriptors, {sum(records[c].nunique() for c in descriptor_cols)} conditions")
pd.DataFrame({"level": desc_level, "distinct values": records[descriptor_cols].nunique()})

9 descriptors, 41 conditions


,level,distinct values
state_id,store,3
cat_id,item,3
dept_id,item,7
item_price_vs_dept_id_cat,item,5
item_price_volatility_cat,item,4
item_launch_cat,item,4
price_level_vs_item_cat,record,5
price_trend_vs_item_cat,record,5
discount_freq_vs_item_cat,record,5


In [7]:
class QualityBase:
    def __init__(self, N, store_codes, min_cov=None, max_cov=None, min_stores=None):
        min_cov = MIN_COV if min_cov is None else min_cov
        max_cov = MAX_COV if max_cov is None else max_cov
        self.N = N
        self.store_codes = store_codes
        self.n_store_codes = int(store_codes.max()) + 1
        self.min_n = max(2, int(np.ceil(min_cov * N)))
        self.max_n = int(np.floor(max_cov * N))
        self.min_stores = MIN_STORES if min_stores is None else min_stores
        self.cache = {}
        self.track = None        # planted mask, if the evaluations should be traced
        self.trace = []          # (time, quality, jaccard with planted, valid)

    def n_stores(self, mask):
        return np.count_nonzero(np.bincount(self.store_codes[mask], minlength=self.n_store_codes))

    def expandable(self, mask):
        return mask.sum() >= self.min_n and self.n_stores(mask) >= self.min_stores

    def satisfies_constraints(self, mask):
        n = mask.sum()
        return n <= self.max_n and n < self.N and self.expandable(mask)

    def optimistic_estimate(self, mask):
        return None

    def __call__(self, mask):
        key = mask.tobytes()
        if key not in self.cache:
            self.cache[key] = self.compute(mask)
            if self.track is not None:
                inter = np.count_nonzero(mask & self.track)
                union = np.count_nonzero(mask | self.track)
                self.trace.append((time.time(), self.cache[key], inter / union, self.satisfies_constraints(mask)))
        return self.cache[key]


class MonetaryQuality(QualityBase):
    def __init__(self, excess, level, store_codes, relative=True, **constraints):
        super().__init__(len(level), store_codes, **constraints)
        self.relative = relative
        self.x = excess.sum(axis=1, dtype=np.float64)
        self.level = level
        self.L_T = level.sum()
        self.y = self.x - level * self.x.sum() / self.L_T

    def signed(self, mask):
        if not self.relative:
            return self.x[mask].sum()
        return self.y[mask].sum() * self.L_T / (self.L_T - self.level[mask].sum())

    def compute(self, mask):
        return abs(self.signed(mask))

    def optimistic_estimate(self, mask):
        v = self.y[mask] if self.relative else self.x[mask]
        bound = max(v[v > 0].sum(), -v[v < 0].sum())
        if self.relative:
            bound *= self.L_T / (self.L_T - self.level[mask].sum())
        return bound


class ClusterQuality(QualityBase):
    def __init__(self, D, store_codes, **constraints):
        super().__init__(len(D), store_codes, **constraints)
        self.D = D
        self.rowsum = D.sum(axis=1, dtype=np.float64)

    def components(self, mask):
        idx = np.flatnonzero(mask)
        n, n_c = len(idx), self.N - len(idx)
        within = self.D[np.ix_(idx, idx)].sum(dtype=np.float64)
        between = self.rowsum[idx].sum() - within
        return between / (n * n_c), within / (n * (n - 1))

    def compute(self, mask):
        alpha, beta = self.components(mask)
        return alpha / (beta + 1)


def _cityblock_block(Xb, X):
    return cdist(Xb, X, metric="cityblock")


def pairwise_cityblock(X, block=512):
    """Full N x N city-block distance matrix, row blocks computed in parallel."""
    N_ = len(X)
    dtype = np.float32 if N_ > 6000 else np.float64
    starts = list(range(0, N_, block))
    parts = Parallel(n_jobs=N_JOBS)(delayed(_cityblock_block)(X[s:s + block], X) for s in starts)
    D = np.empty((N_, N_), dtype=dtype)
    for s, part in zip(starts, parts):
        D[s:s + len(part)] = part
    return D


def update_rows(D, X, rows):
    """Distances after the target rows `rows` changed: only those rows / columns are recomputed."""
    D = D.copy()
    if len(rows):
        new = cdist(X[rows], X, metric="cityblock").astype(D.dtype)
        D[rows, :] = new
        D[:, rows] = new.T
    return D


start = time.time()
D_money = pairwise_cityblock(excess)
D_shape = pairwise_cityblock(rel_dev)
DISTANCE_TIME = time.time() - start
DISTANCE_FOR = {"M1": D_money, "M2": D_money, "M3": D_shape}
print(f"Distance matrices ({N} x {N}, T = {T}, {D_money.dtype}) built in {DISTANCE_TIME:.1f}s")


def make_quality(measure, exc=None, D=None):
    exc = excess if exc is None else exc
    if measure == "M1":
        return MonetaryQuality(exc, level, store_codes, relative=False)
    if measure == "M2":
        return MonetaryQuality(exc, level, store_codes, relative=True)
    return ClusterQuality(D_shape if D is None else D, store_codes)


_phi = make_quality("M2")
_mask = np.zeros(N, dtype=bool)
_mask[: N // 10] = True
_direct = _phi.x[_mask].sum() - level[_mask].sum() * (_phi.x.sum() - _phi.x[_mask].sum()) / (level.sum() - level[_mask].sum())
assert np.isclose(_phi.signed(_mask), _direct)

Distance matrices (6581 x 6581, T = 176, float32) built in 11.3s


In [8]:
def mask_of(A, desc):
    mask = np.ones(len(A), dtype=bool)
    for j, v in desc:
        mask &= A[:, j] == v
    return mask


class ResultSet:
    def __init__(self, q):
        self.q = q
        self.items = {}

    def insert(self, quality, desc, mask):
        key = mask.tobytes()
        old = self.items.get(key)
        if old is not None and len(old[1]) <= len(desc):
            return
        self.items[key] = (quality, desc, mask)
        if len(self.items) > self.q:
            del self.items[min(self.items, key=lambda k: self.items[k][0])]

    def full(self):
        return len(self.items) >= self.q

    def threshold(self):
        return min(v[0] for v in self.items.values()) if self.full() else -np.inf

    def sorted(self):
        return sorted(self.items.values(), key=lambda x: -x[0])


def oe_below_threshold(phi, mask, result_set):
    if not result_set.full():
        return False
    oe = phi.optimistic_estimate(mask)
    return oe is not None and oe < result_set.threshold()


def refine(seed, seed_mask, A, pool=None):
    used = {j for j, _ in seed}
    if pool is None:
        options = ((j, v) for j in range(A.shape[1]) if j not in used for v in pd.unique(A[seed_mask, j]))
    else:
        options = ((j, v) for j, v in pool if j not in used and (A[seed_mask, j] == v).any())
    for condition in options:
        yield tuple(sorted(seed + (condition,), key=lambda c: c[0]))


def _beam(A, phi, pool, w, d, prune, result_set, stats, evaluated_keys, mask_cache):
    def get_mask(desc):
        if desc not in mask_cache:
            mask_cache[desc] = mask_of(A, desc)
        return mask_cache[desc]

    candidate_queue = [()]
    seen, tie = set(), itertools.count()
    for level_ in range(1, d + 1):
        beam = []
        for seed in candidate_queue:
            seed_mask = get_mask(seed)
            if prune and seed and oe_below_threshold(phi, seed_mask, result_set):
                stats["pruned_oe_seed"] += 1
                continue
            for desc in refine(seed, seed_mask, A, pool):
                if desc in seen:
                    continue
                seen.add(desc)
                stats["generated"] += 1
                mask = get_mask(desc)
                if not phi.expandable(mask):
                    stats["skipped_coverage"] += 1
                    continue
                if prune and seed and mask.sum() == seed_mask.sum():
                    stats["pruned_redundant"] += 1
                    continue
                quality = phi(mask)
                stats["visits"] += 1
                evaluated_keys.add(desc)
                if phi.satisfies_constraints(mask):
                    result_set.insert(quality, desc, mask)
                if prune and oe_below_threshold(phi, mask, result_set):
                    stats["pruned_oe"] += 1
                    continue
                item_ = (quality, next(tie), desc)
                if len(beam) < w:
                    heapq.heappush(beam, item_)
                else:
                    heapq.heappushpop(beam, item_)
        candidate_queue = [desc for _, _, desc in sorted(beam, reverse=True)]


def beam_search(A, phi, w=W, d=D_MAX, q=Q, prune=False, pool=None):
    stats, result_set, keys = Counter(), ResultSet(q), set()
    _beam(A, phi, pool, w, d, prune, result_set, stats, keys, {})
    stats["evaluated"] = len(keys)
    return result_set.sorted(), stats


def top_pairs(D, u=U, kind="alpha", block=512):
    """Block-wise pair selection over the lower triangle.
    kind="closest": the u smallest distances (plain BUS, Alg. 2).
    kind="alpha":   the u largest alpha_ij / (D_ij + 1) (alpha-matrix, §4.3.2)."""
    N_ = len(D)
    rowsum = D.sum(axis=1, dtype=np.float64)
    scores, rows_, cols_ = [], [], []
    for s in range(0, N_, block):
        Db = D[s:s + block].astype(np.float64)
        i = np.arange(s, s + len(Db))[:, None]
        j = np.arange(N_)[None, :]
        if kind == "closest":
            score = -Db
        else:
            alpha = (rowsum[i] + rowsum[j] - 2 * Db) / (2 * (N_ - 2))
            score = alpha / (Db + 1)
        score = np.where(j < i, score, -np.inf).ravel()
        k = min(u, int(np.isfinite(score).sum()))
        if k == 0:
            continue
        best = np.argpartition(-score, k - 1)[:k]
        scores.append(score[best])
        rows_.append(s + best // N_)
        cols_.append(best % N_)
    scores, rows_, cols_ = np.concatenate(scores), np.concatenate(rows_), np.concatenate(cols_)
    order = np.argsort(-scores)[:u]
    return [(int(rows_[o]), int(cols_[o])) for o in order]


def shared_conditions(mc, A):
    first = mc[0]
    return [(j, A[first, j]) for j in range(A.shape[1]) if all(A[i, j] == A[first, j] for i in mc)]


def evaluate_candidates(A, phi, candidates, q=Q, prune=False):
    stats = Counter(generated=len(candidates))
    result_set = ResultSet(q)
    dead = set()
    for desc in sorted(candidates, key=len):
        if prune and len(desc) > 1 and any(
            sub in dead for size in range(1, len(desc)) for sub in itertools.combinations(desc, size)
        ):
            stats["pruned_superset"] += 1
            continue
        mask = mask_of(A, desc)
        if not phi.expandable(mask):
            stats["skipped_coverage"] += 1
            dead.add(desc)
            continue
        quality = phi(mask)
        stats["evaluated"] += 1
        stats["visits"] += 1
        if phi.satisfies_constraints(mask):
            result_set.insert(quality, desc, mask)
        if prune and oe_below_threshold(phi, mask, result_set):
            dead.add(desc)
    return result_set.sorted(), stats


def bus(A, phi, pairs, d=D_MAX, q=Q, prune=False):
    candidates = set()
    for mc in pairs:
        conditions = shared_conditions(mc, A)
        for size in range(1, d + 1):
            candidates.update(itertools.combinations(conditions, size))
    top, stats = evaluate_candidates(A, phi, candidates, q, prune)
    stats["mini_clusters"] = len(pairs)
    return top, stats


def ensemble_pool(A, phi, pairs, pool_size=POOL_SIZE, prune=False):
    votes = Counter(c for mc in pairs for c in shared_conditions(mc, A))
    pool = [c for c, _ in votes.most_common(pool_size)]
    top, stats = beam_search(A, phi, prune=prune, pool=pool)
    stats["mini_clusters"] = len(pairs)
    stats["pool"] = len(pool)
    return top, stats, pool


def ensemble_per_mc(A, phi, pairs, w=W_MC, d=D_MAX, q=Q, prune=False):
    stats, result_set, keys, mask_cache = Counter(), ResultSet(q), set(), {}
    spaces = set()
    for mc in pairs:
        space = tuple(shared_conditions(mc, A))
        if not space:
            stats["mc_empty"] += 1
            continue
        if space in spaces:
            stats["mc_duplicate_space"] += 1
            continue
        spaces.add(space)
        _beam(A, phi, list(space), w, d, prune, result_set, stats, keys, mask_cache)
    stats["evaluated"] = len(keys)
    stats["mini_clusters"] = len(pairs)
    stats["mc_spaces"] = len(spaces)
    union = sorted({c for space in spaces for c in space}, key=lambda c: (c[0], str(c[1])))
    return result_set.sorted(), stats, union


def run_method(method, A, phi, D, prune, pairs=None):
    """method: "BS", "BS w=<k>", "BUS", "BUS-α", "BUS→BS pool", "BUS→BS per-MC".
    Runtime includes mini-cluster selection unless precomputed pairs are passed."""
    start = time.time()
    pool = None
    if method.startswith("BS"):
        width = int(method.split("w=")[1]) if "w=" in method else W
        top, stats = beam_search(A, phi, w=width, prune=prune)
    else:
        if pairs is None:
            pairs = top_pairs(D, kind="closest" if method == "BUS" else "alpha")
        stats_pair = time.time() - start
        if method in ("BUS", "BUS-α"):
            top, stats = bus(A, phi, pairs, prune=prune)
        elif method == "BUS→BS pool":
            top, stats, pool = ensemble_pool(A, phi, pairs, prune=prune)
        else:
            top, stats, pool = ensemble_per_mc(A, phi, pairs, prune=prune)
        stats["pair_time"] = stats_pair
    stats["runtime"] = time.time() - start
    return top, stats, pool


def top_keys(top):
    return {m.tobytes() for _, _, m in top}


def jaccard(a, b):
    return len(a & b) / len(a | b) if a | b else 1.0

In [9]:
def permuted_descriptors(rng):
    perm = records[descriptor_cols].copy()
    for key, lvl in (("store", "store"), ("item", "item")):
        cols = [c for c in descriptor_cols if desc_level[c] == lvl]
        if cols:
            table = records.groupby(key)[cols].first()
            donor = dict(zip(table.index, rng.permutation(table.index)))
            perm[cols] = table.loc[records[key].map(donor)].to_numpy()
    for c in [c for c in descriptor_cols if desc_level[c] == "record"]:
        perm[c] = records.groupby("item")[c].transform(lambda s: rng.permutation(s.to_numpy()))
    return perm.astype(str).to_numpy(dtype=object)

In [10]:
POST = np.arange(T) >= BASE_LEN
BUDGETS = [25, 50, 100, 200, 400, 800, 1600, 3200, 6400]
SECONDS = [0.05, 0.1, 0.2, 0.5, 1, 2, 5, 10, 30, 60]


def pattern_curve(kind, delta):
    t = np.arange(T)
    if kind == "level":
        return delta * POST
    if kind == "trend":
        return 2 * delta * np.clip((t - BASE_LEN + 1) / (T - BASE_LEN), 0, None) * POST
    return delta * np.sin(2 * np.pi * t / PERIODS_PER_YEAR)


def n_stores_of(mask):
    return np.count_nonzero(np.bincount(store_codes[mask], minlength=int(store_codes.max()) + 1))


def sample_description(rng, lits, A_, coverage=PLANT_COVERAGE, tries=20000):
    # A random description of `lits` conditions taken from one record, with coverage in range,
    # at least MIN_STORES stores, and every condition actually narrowing the subgroup.
    for _ in range(tries):
        r = int(rng.integers(N))
        cols = rng.choice(A_.shape[1], lits, replace=False)
        desc = tuple(sorted(((int(j), A_[r, j]) for j in cols), key=lambda c: c[0]))
        mask = mask_of(A_, desc)
        if not (coverage[0] <= mask.mean() <= coverage[1]) or n_stores_of(mask) < MIN_STORES:
            continue
        if any(mask_of(A_, tuple(c for c in desc if c != drop)).sum() == mask.sum() for drop in desc):
            continue
        return desc, mask
    return None, None


def sample_interaction(rng, A_, coverage=PLANT_COVERAGE, tries=20000):
    # Conditions A, B on different attributes: A∧B in the coverage range, A∧¬B and ¬A∧B larger than A∧B.
    for _ in range(tries):
        r = int(rng.integers(N))
        j1, j2 = rng.choice(A_.shape[1], 2, replace=False)
        a, b = (int(j1), A_[r, j1]), (int(j2), A_[r, j2])
        mA, mB = mask_of(A_, (a,)), mask_of(A_, (b,))
        mAB = mA & mB
        if not (coverage[0] <= mAB.mean() <= coverage[1]) or n_stores_of(mAB) < MIN_STORES:
            continue
        if (mA & ~mB).sum() < mAB.sum() or (~mA & mB).sum() < mAB.sum():
            continue
        return (a, b), mA, mB, mAB
    return None, None, None, None


def _rank_of_planted(entries):
    # entries: (time, quality, jaccard, valid). Rank of the best planted match among the valid evaluated
    # subgroups (1 = best); NaN if the planted subgroup was not evaluated.
    planted_q = [q for _, q, jac, valid in entries if valid and jac >= HIT_JACCARD]
    if not planted_q:
        return np.nan
    best = max(planted_q)
    return 1 + sum(1 for _, q, jac, valid in entries if valid and jac < HIT_JACCARD and q > best)


def _plant_job(spec, excess, level, D_money, D_shape, store_codes, methods):
    # Builds one planted dataset (null background + planted deviation) and runs every method (pruning on).
    rng = np.random.default_rng(spec["seed"])
    A_ = spec["A"]
    exc = excess.copy()
    changed = []
    for rows, scale in spec["plants"]:
        rows = np.asarray(rows)
        keep = rows[rng.random(len(rows)) >= spec["outliers"]]
        exc[keep] += scale * spec["curve"][None, :] * level[keep, None]
        changed.append(keep)
    changed = np.unique(np.concatenate(changed))
    if spec["measure"] == "M2":
        D = update_rows(D_money, exc, changed)
    else:
        D = update_rows(D_shape, np.clip(exc / level[:, None], -3, 3), changed)
    planted = np.zeros(len(level), dtype=bool)
    planted[spec["target_rows"]] = True

    out = []
    for method in methods:
        if spec["measure"] == "M2":
            phi = MonetaryQuality(exc, level, store_codes, relative=True)
        else:
            phi = ClusterQuality(D, store_codes)
        phi.track, phi.trace = planted, []
        t0 = time.time()
        top, stats, pool = run_method(method, A_, phi, D, True)
        trace = phi.trace
        top_jac = [np.count_nonzero(m & planted) / np.count_nonzero(m | planted) for _, _, m in top]
        reported_ranks = [r + 1 for r, jac in enumerate(top_jac) if jac >= HIT_JACCARD]
        row = {**{k: v for k, v in spec.items() if k not in ("plants", "curve", "target_rows", "A")},
               "method": method, "planted_n": int(planted.sum()),
               "evaluated_hit": any(valid and jac >= HIT_JACCARD for _, _, jac, valid in trace),
               "reported": bool(reported_ranks), "rank": reported_ranks[0] if reported_ranks else np.nan,
               "rank_all": _rank_of_planted(trace), "best_jaccard_top": max(top_jac, default=0.0),
               "evaluated": len(trace), "runtime_s": stats["runtime"], "best_quality": top[0][0] if top else np.nan}
        for b in BUDGETS:
            row[f"rank_at_{b}"] = _rank_of_planted(trace[:b])
        for s in SECONDS:
            row[f"rank_at_{s}s"] = _rank_of_planted([e for e in trace if e[0] - t0 <= s])
        out.append(row)
    return out


def run_plant_specs(specs, methods):
    jobs = max(1, min(N_JOBS, len(specs)))
    parts = Parallel(n_jobs=jobs)(
        delayed(_plant_job)(spec, excess, level, D_money, D_shape, store_codes, methods) for spec in specs)
    return pd.DataFrame([row for part in parts for row in part])

In [11]:
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components

expected = S - excess
POST = np.arange(T) >= BASE_LEN
SIN = np.sin(2 * np.pi * np.arange(T) / PERIODS_PER_YEAR)


class VolatilityQuality(QualityBase):
    """M4: how much more (or less) the subgroup's deviation from its peers fluctuates after the base window.
    v_i = log(var post / var base) of r_i(t); quality = |mean v in S - mean v in complement| * sqrt(n n_c / N)."""

    def __init__(self, rel, store_codes, **constraints):
        super().__init__(len(rel), store_codes, **constraints)
        self.v = np.log((rel[:, BASE_LEN:].var(axis=1) + 1e-6) / (rel[:, :BASE_LEN].var(axis=1) + 1e-6))
        self.total = self.v.sum()

    def compute(self, mask):
        n = mask.sum()
        s = self.v[mask].sum()
        return abs(s / n - (self.total - s) / (self.N - n)) * np.sqrt(n * (self.N - n) / self.N)


def block_means(rel):
    post = rel[:, BASE_LEN:]
    k = max(1, post.shape[1] // N_BLOCKS)
    m = (post.shape[1] // k) * k
    return post[:, :m].reshape(len(rel), -1, k).mean(axis=2)


def apply_pattern(kind, rows, strength, rng, excess, expected, level):
    """Returns the planted excess. strength: share of normal sales (level/trend/season), share of post-base
    periods zeroed (stockout), extra fluctuation (volatility) or periods of lag (lag)."""
    exc = excess.copy()
    if kind == "level":
        exc[rows] += strength * POST[None, :] * level[rows, None]
    elif kind == "trend":
        ramp = np.clip((np.arange(T) - BASE_LEN + 1) / (T - BASE_LEN), 0, None) * POST
        exc[rows] += 2 * strength * ramp[None, :] * level[rows, None]
    elif kind == "season":
        exc[rows] += strength * SIN[None, :] * level[rows, None]
    elif kind == "stockout":
        actual = exc[rows] + expected[rows]
        lost = (rng.random(actual.shape) < strength) & POST[None, :]
        exc[rows] = np.where(lost, 0.0, actual) - expected[rows]
    elif kind == "lag":
        actual = exc[rows] + expected[rows]
        k = int(strength)
        shifted = np.concatenate([np.repeat(actual[:, :1], k, axis=1), actual[:, :-k]], axis=1)
        exc[rows] = shifted - expected[rows]
    elif kind == "volatility":
        post = exc[rows][:, POST]
        mean = post.mean(axis=1, keepdims=True)
        block = exc[rows]
        block[:, POST] = mean + (1 + strength) * (post - mean)
        exc[rows] = block
    return exc


# ---------------- seeds (mini-clusters)
def knn_lists(D, k=KNN, block=1024):
    out = np.empty((len(D), k), dtype=np.int64)
    for s in range(0, len(D), block):
        Db = np.array(D[s:s + block], dtype=np.float32)
        Db[np.arange(len(Db)), np.arange(s, s + len(Db))] = np.inf
        part = np.argpartition(Db, k, axis=1)[:, :k]
        out[s:s + len(Db)] = part
    return out


def snn_edges(D):
    knn = knn_lists(D)
    sets = [set(r) for r in knn.tolist()]
    edges = []
    for i, row in enumerate(knn.tolist()):
        for j in row:
            if j > i and i in sets[j]:
                edges.append((len(sets[i] & sets[j]), float(D[i, j]), i, j))
    return edges


def seeds_snn_pairs(D, u=U_SEEDS):
    edges = sorted(snn_edges(D), key=lambda e: (-e[0], e[1]))
    return [(i, j) for _, _, i, j in edges[:u]]


def seeds_snn_components(D, u=U_SEEDS):
    edges = [e for e in snn_edges(D) if e[0] >= SNN_MIN]
    if not edges:
        return []
    i = np.array([e[2] for e in edges])
    j = np.array([e[3] for e in edges])
    graph = coo_matrix((np.ones(len(i)), (i, j)), shape=(len(D), len(D)))
    _, labels = connected_components(graph, directed=False)
    comps = pd.Series(np.arange(len(D))).groupby(labels).apply(list)
    max_size = max(50, int(0.05 * len(D)))
    comps = [tuple(c) for c in comps if Z_COMP_MIN <= len(c) <= max_size]
    return sorted(comps, key=len, reverse=True)[:u]


def conditions_of(mc, A_, rule):
    if rule == "all":
        return shared_conditions(mc, A_)
    sub = A_[list(mc)]
    conds = []
    for j in range(A_.shape[1]):
        values, counts = np.unique(sub[:, j].astype(str), return_counts=True)
        best = counts.argmax()
        if counts[best] >= MAJORITY * len(mc):
            conds.append((j, values[best]))
    return conds


def candidates_of(mcs, A_, rule):
    cands = set()
    for mc in mcs:
        conds = conditions_of(mc, A_, rule)
        for size in range(1, D_MAX + 1):
            cands.update(itertools.combinations(conds, size))
    return cands


def pool_of(mcs, A_, rule, pool_size=POOL_SIZE):
    votes = Counter(c for mc in mcs for c in conditions_of(mc, A_, rule))
    return [c for c, _ in votes.most_common(pool_size)]


# ---------------- Q3: local search from BUS candidates
def local_search(A_, phi, starts, result_set, stats, max_rounds=10):
    """Hill climbing in the lattice from the best BUS candidates: drop one condition, add one condition
    (values present in the subgroup), or swap the value of one condition; move while quality improves."""
    frontier = [desc for _, desc, _ in starts[:LOCAL_TOPK]]
    seen = set(frontier)
    for _ in range(max_rounds):
        nxt = []
        for desc in frontier:
            mask = mask_of(A_, desc)
            best_q, best_d = phi(mask) if phi.expandable(mask) else -np.inf, None
            neighbours = []
            if len(desc) > 1:
                neighbours += [tuple(c for c in desc if c != drop) for drop in desc]
            used = {j for j, _ in desc}
            if len(desc) < D_MAX:
                for j in range(A_.shape[1]):
                    if j not in used:
                        neighbours += [tuple(sorted(desc + ((j, v),), key=lambda c: c[0])) for v in pd.unique(A_[mask, j])]
            for c in desc:
                rest = tuple(x for x in desc if x != c)
                for v in pd.unique(A_[mask_of(A_, rest), c[0]]):
                    if v != c[1]:
                        neighbours.append(tuple(sorted(rest + ((c[0], v),), key=lambda x: x[0])))
            for nb in neighbours:
                if nb in seen:
                    continue
                seen.add(nb)
                m = mask_of(A_, nb)
                if not phi.expandable(m):
                    continue
                q = phi(m)
                stats["local_evaluated"] += 1
                if phi.satisfies_constraints(m):
                    result_set.insert(q, nb, m)
                if q > best_q:
                    best_q, best_d = q, nb
            if best_d is not None:
                nxt.append(best_d)
        frontier = nxt
        if not frontier:
            break
    return result_set.sorted()


def _rank_of_planted(entries):
    planted_q = [q for _, q, jac, valid in entries if valid and jac >= HIT_JACCARD]
    if not planted_q:
        return np.nan
    best = max(planted_q)
    return 1 + sum(1 for _, q, jac, valid in entries if valid and jac < HIT_JACCARD and q > best)


BUDGETS = [25, 50, 100, 200, 400, 800, 1600, 3200]
SECONDS = [0.1, 0.25, 0.5, 1, 2, 5, 10, 30]

In [12]:
def _task_job(spec, excess, expected, level, D_money, D_shape, D_blocks, store_codes, methods):
    """One planted task: plant the pattern, build every seed type (timed), record seed diagnostics,
    then run every method with a traced quality function (pruning on)."""
    rng = np.random.default_rng(spec["seed"])
    A_ = spec["A"]
    rows = spec["target_rows"]
    planted = np.zeros(len(level), dtype=bool)
    planted[rows] = True
    exc = apply_pattern(spec["pattern"], rows, spec["strength"], rng, excess, expected, level)
    rel = np.clip(exc / level[:, None], -3, 3)

    t = time.time(); Dm = update_rows(D_money, exc, rows); t_money = time.time() - t
    t = time.time(); Ds = update_rows(D_shape, rel, rows); t_shape = time.time() - t
    t = time.time(); Db = update_rows(D_blocks, block_means(rel), rows); t_blocks = time.time() - t

    # ---- seeds
    seed_builders = {
        "closest-$": (lambda: top_pairs(Dm, U_SEEDS, "closest"), "all", t_money),
        "alpha-$": (lambda: top_pairs(Dm, U_SEEDS, "alpha"), "all", t_money),
        "closest-rel": (lambda: top_pairs(Ds, U_SEEDS, "closest"), "all", t_shape),
        "closest-blocks": (lambda: top_pairs(Db, U_SEEDS, "closest"), "all", t_blocks),
        "snn-pairs": (lambda: seeds_snn_pairs(Db), "all", t_blocks),
        "snn-components": (lambda: seeds_snn_components(Db), "majority", t_blocks),
    }
    seeds, seed_rows = {}, []
    meta = {k: v for k, v in spec.items() if k not in ("A", "target_rows")}
    for name, (build, rule, dist_time) in seed_builders.items():
        t = time.time()
        mcs = build()
        cands = candidates_of(mcs, A_, rule)
        seed_time = time.time() - t + dist_time
        seeds[name] = (mcs, rule, cands, seed_time)
        members = [list(mc) for mc in mcs]
        purity = np.mean([planted[m].mean() >= 0.5 for m in members]) if members else 0.0
        covered = np.unique(np.concatenate(members)) if members else np.array([], dtype=int)
        recall = any(np.count_nonzero(mask_of(A_, c) & planted) / np.count_nonzero(mask_of(A_, c) | planted) >= HIT_JACCARD
                     for c in cands if len(c) <= spec["lits"] + 1) if cands else False
        seed_rows.append({**meta, "seed": name, "rule": rule, "mini_clusters": len(mcs),
                          "mean_size": np.mean([len(m) for m in members]) if members else 0,
                          "distinct_records": len(covered),
                          "purity": purity, "purity_if_random": planted.mean(),
                          "planted_coverage": planted[covered].sum() / planted.sum() if len(covered) else 0.0,
                          "size_percentile": pd.Series(level).rank(pct=True).to_numpy()[covered].mean() if len(covered) else np.nan,
                          "candidates": len(cands), "candidate_recall": recall, "seed_time_s": seed_time})

    # ---- methods
    def make_phi():
        if spec["measure"] == "M2":
            return MonetaryQuality(exc, level, store_codes, relative=True)
        if spec["measure"] == "M3":
            return ClusterQuality(Ds, store_codes)
        return VolatilityQuality(rel, store_codes)

    out = []
    for method in methods:
        phi = make_phi()
        phi.track, phi.trace = planted, []
        stats = Counter()
        offset = 0.0
        t0 = time.time()
        if method.startswith("BS"):
            width = int(method.split("w=")[1]) if "w=" in method else W
            top, stats = beam_search(A_, phi, w=width, prune=True)
        elif method.startswith("BUS→BS pool"):
            key = {"α-$": "alpha-$", "blocks": "closest-blocks", "snn": "snn-pairs"}[method.split("pool ")[1]]
            mcs, rule, _, offset = seeds[key]
            top, stats = beam_search(A_, phi, prune=True, pool=pool_of(mcs, A_, rule))
        else:
            key = method.split(" ")[1]
            mcs, rule, cands, offset = seeds[key]
            top, stats = evaluate_candidates(A_, phi, cands, prune=True)
            if method.endswith("+local"):
                result_set = ResultSet(Q)
                for q_, d_, m_ in top:
                    result_set.insert(q_, d_, m_)
                top = local_search(A_, phi, top, result_set, stats)
        runtime = time.time() - t0 + offset
        trace = [(tt - t0 + offset, q_, jac, valid) for tt, q_, jac, valid in phi.trace]
        top_jac = [np.count_nonzero(m & planted) / np.count_nonzero(m | planted) for _, _, m in top]
        ranks = [r + 1 for r, jac in enumerate(top_jac) if jac >= HIT_JACCARD]
        row = {**meta, "method": method, "planted_n": int(planted.sum()),
               "evaluated_hit": any(valid and jac >= HIT_JACCARD for _, _, jac, valid in trace),
               "rank": ranks[0] if ranks else np.nan, "rank_all": _rank_of_planted(trace),
               "evaluated": len(trace), "runtime_s": runtime, "seed_time_s": offset,
               "local_evaluated": stats.get("local_evaluated", 0)}
        for b in BUDGETS:
            row[f"rank_at_{b}"] = _rank_of_planted(trace[:b])
        for s in SECONDS:
            row[f"rank_at_{s}s"] = _rank_of_planted([e for e in trace if e[0] <= s])
        out.append(row)
    return out, seed_rows

In [13]:
# ---------------- calibration: null spread of random described subgroups (as in v3)
rng = np.random.default_rng(SEED + 4)
rates, amps, vols = [], [], []
v_all = np.log((rel_dev[:, BASE_LEN:].var(axis=1) + 1e-6) / (rel_dev[:, :BASE_LEN].var(axis=1) + 1e-6))
for _ in range(20):
    A_null = permuted_descriptors(rng)
    for _ in range(10):
        desc, mask = sample_description(rng, int(rng.choice(LITS_V4)), A_null)
        if desc is not None:
            rates.append(excess[mask][:, POST].sum() / (level[mask].sum() * POST.sum()))
            amps.append(2 * (rel_dev[mask].mean(axis=0) * SIN).mean())
            vols.append(v_all[mask].mean())
SIGMA = {"rate": float(np.std(rates)), "season": float(np.std(amps)), "vol": float(np.std(vols))}
print("null spread:", {k: round(v, 4) for k, v in SIGMA.items()})


def strength_of(pattern, snr):
    if pattern in ("level", "trend"):
        return snr * SIGMA["rate"]
    if pattern == "stockout":
        return min(0.9, snr * SIGMA["rate"])
    if pattern == "season":
        return snr * SIGMA["season"]
    if pattern == "volatility":
        return float(np.exp(snr * SIGMA["vol"] / 2) - 1)
    return LAG_STEPS[snr] * LAG_UNIT


# ---------------- base matrix of the denoised representation
start = time.time()
D_blocks = pairwise_cityblock(block_means(rel_dev))
print(f"block distance matrix in {time.time() - start:.1f}s")

# ---------------- tasks
level_pct = pd.Series(level).rank(pct=True).to_numpy()
specs = []
for pattern, measure in PATTERNS_V4.items():
    for snr in SNR_V4:
        for lits in LITS_V4:
            for rep in range(REPS_V4):
                A_null = permuted_descriptors(rng)
                desc, mask = sample_description(rng, lits, A_null)
                if desc is None:
                    continue
                rows = np.flatnonzero(mask)
                levels = Counter(desc_level[descriptor_cols[j]] for j, _ in desc)
                specs.append({"dataset": DATASET, "pattern": pattern, "measure": measure, "snr": snr,
                              "strength": strength_of(pattern, snr), "lits": lits, "rep": rep,
                              "description": describe(desc), "coverage": mask.mean(),
                              "planted_size_pct": level_pct[rows].mean(),
                              "store_conds": levels.get("store", 0), "item_conds": levels.get("item", 0),
                              "record_conds": levels.get("record", 0),
                              "seed": int(rng.integers(1 << 31)), "A": A_null, "target_rows": rows})
print(f"{len(specs)} planted tasks x {len(METHODS_V4)} methods")

start = time.time()
jobs = max(1, min(N_JOBS, len(specs)))
parts = Parallel(n_jobs=jobs)(
    delayed(_task_job)(spec, excess, expected, level, D_money, D_shape, D_blocks, store_codes, METHODS_V4)
    for spec in specs)
tasks_df = pd.DataFrame([r for out, _ in parts for r in out])
seeds_df = pd.DataFrame([r for _, rows_ in parts for r in rows_])
tasks_df.to_csv(f"{RESULTS_DIR}/{DATASET}_tasks.csv", index=False)
seeds_df.to_csv(f"{RESULTS_DIR}/{DATASET}_seeds.csv", index=False)
print(f"done in {time.time() - start:.0f}s")

# ---------------- dataset meta-features (Q2)
post = rel_dev[:, BASE_LEN:]
centered = post - post.mean(axis=1, keepdims=True)
ac1 = (centered[:, 1:] * centered[:, :-1]).sum(axis=1) / np.maximum((centered ** 2).sum(axis=1), 1e-12)
season_r2 = np.corrcoef(rel_dev.mean(axis=0), SIN)[0, 1] ** 2
meta_df = pd.DataFrame([{
    "dataset": DATASET, "N": N, "T": T, "period": PERIOD, "n_descriptors": len(descriptor_cols),
    "n_conditions": int(sum(records[c].nunique() for c in descriptor_cols)),
    "store_level_share": np.mean([desc_level[c] == "store" for c in descriptor_cols]),
    "n_stores": records["store"].nunique(), "noise_rel_std": float(np.median(post.std(axis=1))),
    "autocorr_lag1": float(np.median(ac1)), "seasonality_r2": float(season_r2),
    "intermittency": float((S == 0).mean()), "sigma_rate": SIGMA["rate"], "sigma_season": SIGMA["season"],
    "sigma_vol": SIGMA["vol"]}])
meta_df.to_csv(f"{RESULTS_DIR}/{DATASET}_meta.csv", index=False)
display(meta_df.round(4))

null spread: {'rate': 0.0398, 'season': 0.0095, 'vol': 0.081}
block distance matrix in 1.2s
108 planted tasks x 12 methods
done in 1242s


,dataset,N,T,period,n_descriptors,n_conditions,store_level_share,n_stores,noise_rel_std,autocorr_lag1,seasonality_r2,intermittency,sigma_rate,sigma_season,sigma_vol
0,m5_weekly,6581,176,week,9,41,0.1111,10,0.4456,0.371,0.0002,0.1079,0.0398,0.0095,0.081


In [14]:
tasks_df["found"] = tasks_df["rank"] <= Q
print("Q4 seed diagnostics (mean over tasks)")
display(seeds_df.groupby("seed", sort=False)[["mini_clusters", "mean_size", "distinct_records", "purity", "purity_if_random",
                                              "planted_coverage", "size_percentile", "candidates", "candidate_recall",
                                              "seed_time_s"]].mean().round(3))
print("Q4 / Q1: share of planted subgroups in the top-20, per pattern")
display(tasks_df.pivot_table(index="pattern", columns="method", values="found", sort=False)[METHODS_V4].round(2))
print("overall")
display(tasks_df.groupby("method", sort=False).agg(found=("found", "mean"), generated=("evaluated_hit", "mean"),
                                                   evaluated=("evaluated", "mean"), runtime_s=("runtime_s", "mean"),
                                                   seed_time_s=("seed_time_s", "mean"))
        .assign(**{f"top20@{b}": tasks_df.groupby("method", sort=False)[f"rank_at_{b}"].apply(lambda s: np.mean(s <= Q)) for b in [100, 400]})
        .reindex(METHODS_V4).round(3))

Q4 seed diagnostics (mean over tasks)
Q4 / Q1: share of planted subgroups in the top-20, per pattern
overall


,mini_clusters,mean_size,distinct_records,purity,purity_if_random,planted_coverage,size_percentile,candidates,candidate_recall,seed_time_s
seed,,,,,,,,,,
closest-$,200.00,2.000,37.250,0.064,0.046,0.004,0.009,580.806,0.194,4.408
alpha-$,200.00,2.000,38.111,0.064,0.046,0.004,0.009,586.444,0.194,6.554
closest-rel,200.00,2.000,114.806,0.033,0.046,0.010,0.922,677.713,0.241,4.483
closest-blocks,200.00,2.000,159.380,0.034,0.046,0.012,0.826,909.991,0.333,3.536
snn-pairs,200.00,2.000,224.944,0.076,0.046,0.034,0.684,1094.074,0.694,1.989
snn-components,78.63,4.997,392.583,0.020,0.046,0.059,0.647,1368.796,0.630,2.504


method,BS w=10,BS,BUS closest-$,BUS→BS pool α-$,BUS closest-rel,BUS closest-blocks,BUS snn-pairs,BUS snn-components,BUS→BS pool blocks,BUS→BS pool snn,BUS snn-pairs +local,BUS closest-blocks +local
pattern,,,,,,,,,,,,
level,0.44,0.39,0.22,0.22,0.22,0.28,0.39,0.39,0.39,0.39,0.39,0.39
trend,0.61,0.61,0.11,0.28,0.22,0.11,0.61,0.61,0.50,0.61,0.61,0.56
stockout,0.39,0.44,0.00,0.22,0.00,0.06,0.22,0.22,0.22,0.39,0.44,0.28
season,0.06,0.00,0.00,0.00,0.00,0.00,0.06,0.00,0.00,0.00,0.06,0.00
lag,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
volatility,0.78,0.78,0.11,0.44,0.17,0.22,0.56,0.39,0.67,0.78,0.72,0.72


,found,generated,evaluated,runtime_s,seed_time_s,top20@100,top20@400
method,,,,,,,
BS w=10,0.380,0.556,361.991,2.870,0.000,0.148,0.361
BS,0.370,0.815,816.213,6.023,0.000,0.148,0.259
BUS closest-$,0.074,0.194,457.593,7.928,4.408,0.046,0.065
BUS→BS pool α-$,0.194,0.454,641.491,11.362,6.554,0.056,0.204
BUS closest-rel,0.102,0.241,533.583,8.283,4.483,0.037,0.093
BUS closest-blocks,0.111,0.333,684.907,7.978,3.536,0.056,0.120
BUS snn-pairs,0.306,0.694,790.593,6.838,1.989,0.065,0.250
BUS snn-components,0.269,0.630,958.769,8.022,2.504,0.074,0.194
BUS→BS pool blocks,0.296,0.657,746.880,8.809,3.536,0.120,0.269
